# GQA (Grouped Query Attention)

GQA 是 **Multi-Head Attention (MHA)** 的改进版本，核心思想是：

1. 将 **QKV 矩阵按维度分组**（grouped），减少计算量  
2. 引入 **Query 分组机制**，明确每组 Query 与 Key/Value 的对应关系  
3. 对长文本处理更高效，同时保持建模能力  

---

## 1. 输入与分组设定

- 输入序列：$X \in \mathbb{R}^{L \times d}$ （长度 $L$，隐藏维度 $d$）  
- 分组数：$G$  
- 每组维度：$d_g = d / G$  

---

## 2. 线性投影

生成 Query、Key、Value：

$$
Q = W_Q X, \quad K = W_K X, \quad V = W_V X
$$

---

## 3. Query 与 Key/Value 分组关系

### 一对一（One-to-One）模式

- 每组 Query 只与对应组 Key/Value 计算注意力  
- 对第 $g$ 组 Query：

$$
Q^{(g)} \in \mathbb{R}^{L \times d_g}, \quad 
K^{(g)}, V^{(g)} \in \mathbb{R}^{L \times d_g}
$$

- 注意力计算：

$$
\text{Attention}^{(g)} = \text{Softmax}\Big(\frac{Q^{(g)} {K^{(g)}}^\top}{\sqrt{d_g}}\Big) V^{(g)}
$$

---

### 一对多（One-to-Many）模式

- 某组 Query 可以与 **多组 Key/Value** 进行注意力计算  
- 对第 $g$ 组 Query 与一系列 Key/Value $\{K^{(h)}, V^{(h)}\}$：

$$
\text{Attention}^{(g)} = \sum_{h \in \text{GroupMap}(g)} 
\text{Softmax}\Big(\frac{Q^{(g)} {K^{(h)}}^\top}{\sqrt{d_h}}\Big) V^{(h)}
$$

> 这里 `GroupMap(g)` 定义了 Query 组 g 对应的 Key/Value 组索引集合，清楚表达多对一/一对多关系。

---

## 4. 合并输出

将各组 Attention 输出沿维度拼接：

$$
\text{GQA}(X) = [\text{Attention}^{(1)}, \dots, \text{Attention}^{(G)}] W_O
$$

- $W_O \in \mathbb{R}^{d \times d}$ 是输出投影矩阵  

---

## 5. 总结特点

| 特点 | 说明 |
|------|------|
| Query 分组 | 可以配置一对一或一对多注意力关系，灵活控制计算量 |
| QKV 分组 | 降低 $d \times d$ 线性映射计算成本 |
| 高效长文本 | 分组机制保证计算量随组数减少，适合长序列 |
| 与 MHA 兼容 | 可以直接替换标准 MHA，多头并行依旧可用 |

---

## 6. 公式汇总

$$
\text{GQA}(X) = \Big[
\text{Attention}^{(1)}, \dots, \text{Attention}^{(G)}
\Big] W_O
$$
$$
\text{其中，一对一模式: } 
\text{Attention}^{(g)} = \text{Softmax}\Big(\frac{Q^{(g)} {K^{(g)}}^\top}{\sqrt{d_g}}\Big) V^{(g)}

\text{一对多模式: }
\text{Attention}^{(g)} = \sum_{h \in \text{GroupMap}(g)} 
\text{Softmax}\Big(\frac{Q^{(g)} {K^{(h)}}^\top}{\sqrt{d_h}}\Big) V^{(h)}
$$

In [9]:
import torch
import torch.nn as nn
from transformers.processing_utils import  Unpack
from transformers.utils import  TransformersKwargs
from transformers.cache_utils import Cache, DynamicCache
from collections.abc import Callable
import sys
import os
from typing import  Optional
sys.path.append("/home/jx/experiment/BedRock-llm") # 修改为本地路径
from model import BedrockV3Config
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS

In [6]:
class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states: torch.Tensor):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        hidden_states = self.weight * hidden_states.to(input_dtype)
        return hidden_states


In [3]:
def repeat_kv(hidden_states: torch.Tensor, n_rep: int):
    """
    This is the equivalent of torch.repeat_interleave(hidden_states, n_rep, dim=1), the hidden states go from (batch, num_key_value_heads, seqlen, head_dim)
    to (batch, num_attention_heads, seqlen, head_dim)
    """
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)

In [8]:
def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    cos = cos.unsqueeze(unsqueeze_dim) # [batch,1,seq_len,dim]
    sin = sin.unsqueeze(unsqueeze_dim) # [batch,1,seq_len,dim]
    q_embed = (q * cos) + (rotate_half(q) * sin) # 这里q*cos，在q的dim维度上全部乘以cos, rotate_half(q) * sin,在q的dim维度上全部乘以sin,然后相加对应公式f_q(x_m, m)
    k_embed = (k * cos) + (rotate_half(k) * sin) # 这里k*cos，在k的dim维度上全部乘以cos, rotate_half(k) * sin,在k的dim维度上全部乘以sin,然后相加对应公式f_k(x_m, m)
    return q_embed, k_embed

In [4]:
def eager_attention_forward(
    module: nn.Module,
    query: torch.Tensor,
    key: torch.Tensor,
    value: torch.Tensor,
    attention_mask: torch.Tensor | None,
    scaling: float,
    dropout: float = 0.0,
    **kwargs: Unpack[TransformersKwargs]
):
    key_states = repeat_kv(key, module.num_key_value_heads) # (batch, num_attention_heads, seqlen, head_dim)
    value_states = repeat_kv(value, module.num_key_value_heads) 

    atten_weight = torch.matmul(query, key_states.transpose(2,3)) * scaling # [batch, num_attention_heads, seqlen, seqlen]
    if attention_mask is not None:
        atten_weight = atten_weight + attention_mask # this is causal mask

    attn_weights = nn.functional.softmax(attn_weights, dim=-1, dtype=torch.float32).to(query.dtype)
    attn_weights = nn.functional.dropout(attn_weights, p=dropout, training=module.training)
    attn_output = torch.matmul(attn_weights, value_states) # [batch, num_attention_heads, seqlen, head_dim]
    attn_output = attn_output.transpose(1, 2).contiguous() # [batch, seqlen, num_attention_heads, head_dim]

    return attn_output, attn_weights




In [ ]:
class Attention(nn.Module):
    def __init__(self, config: BedrockV3Config, layer_idx: int):
        super().__init__()

        self.config = config
        self.layer_idx = layer_idx
        self.head_dim = getattr(config, "head_dim", config.hidden_size // config.num_attention_heads)
        self.num_key_value_groups = config.num_attention_heads // config.num_key_value_heads # 多少个组 Key/Value
        self.scaling = self.head_dim ** -0.5
        self.attention_dropout = config.attention_dropout
        self.is_causal = True

        self.q_proj = nn.Linear(
            config.hidden_size, config.num_attention_heads * self.head_dim, bias=config.attention_bias
        )
        self.k_proj = nn.Linear(
            config.hidden_size, config.num_key_value_heads * self.head_dim, bias=config.attention_bias
        )
        self.v_proj = nn.Linear(
            config.hidden_size, config.num_key_value_heads * self.head_dim, bias=config.attention_bias
        )
        self.o_proj = nn.Linear(
            config.num_attention_heads * self.head_dim, config.hidden_size, bias=config.attention_bias
        )

        self.q_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)
        self.k_norm = RMSNorm(self.head_dim, eps=config.rms_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: torch.Tensor | None,
        past_key_value: Cache | None = None,
        cache_position: torch.LongTensor | None = None,
        **kwargs: Unpack[TransformersKwargs]
    ) -> tuple[torch.Tensor, Cache | None]:

        input_shape = hidden_states.shape[:-1] # (batch, seqlen)
        hidden_shape = (*input_shape, -1, self.head_dim) # (batch, seqlen, -1, head_dim)

        query_states = self.q_norm(self.q_proj(hidden_states).view(hidden_shape)).transpose(1,2) # (batch, num_attention_heads, seqlen, head_dim)
        key_states = self.k_norm(self.k_proj(hidden_states).view(hidden_shape)).transpose(1,2)
        value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1,2)

        cos, sin = position_embeddings # [batch,seq_len,dim]
        query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

        if past_key_value is not None:
            # sin and cos are specific to RoPE modules, cache_position needed for the static cache
            cache_kwargs = {"sin": sin, "cos": cos, "cache_position": cache_position}
            # The expected shape for each tensor in the `CacheLayer`s is `[batch_size, num_heads, seq_len, head_dim]`.
            key_states, value_states = past_key_value.update(key_states, value_states, self.layer_idx, cache_kwargs)# 这里更新key_states和value_states
        
        attention_interface = Callable = ALL_ATTENTION_FUNCTIONS.get_interface(
            self.config._attn_implementation, eager_attention_forward
        )

        attn_output, attn_weights = attention_interface(
            self,
            query_states,
            key_states,
            value_states,
            attention_mask,
            dropout=0.0 if not self.training else self.attention_dropout,
            scaling=self.scaling,
            # sliding_window=self.sliding_window,  # we delete this funciton
            **kwargs,
        )
        
        attn_output = attn_output.reshape(*input_shape, -1).contiguous() # (batch, seqlen, dim)
        attn_output = self.o_proj(attn_output)

        return attn_output, attn_weights
